In [6]:
from langgraph.graph import StateGraph,END,START
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
from typing import TypedDict
import os
from langchain_groq import ChatGroq

In [ ]:
load_dotenv(dotenv_path=".env", override=True)

True

In [4]:
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

In [5]:
model = ChatGroq(
    model="llama-3.3-70b-versatile",  
    api_key=GROQ_API_KEY  
)

In [10]:
class jokestate(TypedDict):
    title:str
    jokes:str
    explaination:str


In [9]:
graph=StateGraph(jokestate)

In [11]:
def joke(state:jokestate)->jokestate:
    prompt=f"Generate the joke of the title{state['title']} in a funny way."
    response=model.invoke(prompt).content
    return {'jokes':response}

In [12]:
def explaination(state:jokestate)->jokestate:
    prompt=f"Explain the joke {state['jokes']}of the title{state['title']} in a funny way."
    response=model.invoke(prompt).content
    return {'explaination':response}

In [13]:
graph.add_node('joke',joke)
graph.add_node('explaination',explaination)

In [14]:
graph.add_edge(START,'joke')
graph.add_edge('joke','explaination')
graph.add_edge('explaination',END)
checkpoint=InMemorySaver()
workflow=graph.compile(checkpointer=checkpoint)

In [ ]:
config1={}